# M12. 정규화 / 표준화

> 📌 **언제 필요한가**  
> 변수들의 단위/크기가 달라서 모델이 학습 못 할 때.  
> 예: 키(170)와 몸무게(65)가 모델에선 비슷한 크기여야 함.

## 이 모듈에서 배울 것

- 왜 정규화가 필요한가
- 표준화 (z-score): `(x - 평균) / 표준편차`
- 최소-최대 정규화: `(x - min) / (max - min)`
- 4차시 회귀 학습에서 본 그것 한 번 더 정리

---


## 1. 왜 정규화?

펭귄 데이터를 떠올려봅시다. 4차시에서 봤듯이:
- 지느러미 길이: 약 200 mm
- 부리 길이: 약 50 mm  
- 몸무게: 약 4000 g

숫자 크기가 너무 달라요. 모델이 학습할 때 큰 값에 휘둘려요. 그래서 **다 비슷한 크기로 맞춰주는 게 정규화**.


## 2. 두 가지 방법

### 방법 1: **표준화 (z-score)**

$$z = \frac{x - \mu}{\sigma}$$

평균을 0, 표준편차를 1로 맞춤. 머신러닝에서 가장 흔함.


In [ ]:
import pandas as pd
import numpy as np

# 예시 데이터
df = pd.DataFrame({
    '키': [170, 175, 180, 165, 172],
    '몸무게': [65, 75, 85, 55, 70],
    '나이': [20, 25, 30, 22, 28]
})
print("원본:")
print(df)


In [ ]:
# 표준화
def standardize(s):
    return (s - s.mean()) / s.std()

df_std = df.apply(standardize)
print("표준화 후:")
print(df_std.round(3))
print()
print(f"평균: {df_std.mean().round(3).tolist()}")
print(f"표준편차: {df_std.std().round(3).tolist()}")


**확인**: 평균이 0, 표준편차가 1. 완벽한 표준화.


### 방법 2: **최소-최대 정규화**

$$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$

모든 값을 [0, 1] 범위로. 이미지 처리 등에서 흔함.


In [ ]:
def min_max(s):
    return (s - s.min()) / (s.max() - s.min())

df_mm = df.apply(min_max)
print("최소-최대 정규화 후:")
print(df_mm.round(3))
print()
print(f"최솟값: {df_mm.min().tolist()}")
print(f"최댓값: {df_mm.max().tolist()}")


## 3. 언제 어떤 걸 쓰나?

| 상황 | 추천 |
|---|---|
| 일반 머신러닝 회귀/분류 | **표준화 (z-score)** |
| 신경망 입력 | 표준화 또는 최소-최대 |
| 이상치 많음 | **표준화** (덜 민감) |
| 범위가 의미 있음 (예: 색상 0-255) | 최소-최대 |


## 4. sklearn으로 — `StandardScaler`, `MinMaxScaler`

수작업 대신 sklearn 활용 (테스트 데이터 같은 단위로 변환할 때 편함):


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

scaler = StandardScaler()
df_std_sk = pd.DataFrame(
    scaler.fit_transform(df),
    columns=df.columns
)
print("StandardScaler 결과:")
print(df_std_sk.round(3))


## 5. 본인 데이터에 적용해보기 ✏️


In [ ]:
# 모델 학습 전에 수치 컬럼들 표준화
# 
# # 수치 컬럼만 골라서
# numeric_cols = my_df.select_dtypes(include='number').columns
# 
# # 표준화
# for col in numeric_cols:
#     my_df[col] = (my_df[col] - my_df[col].mean()) / my_df[col].std()


## 6. ⚠️ 함정 / 주의사항

### 6.1 데이터 누수 (data leakage)
훈련/테스트 데이터 나눈 후, **훈련 데이터의 평균/표준편차로** 둘 다 변환.  
**잘못된 방법**: 전체 데이터로 평균 계산 → 테스트 정보가 훈련에 새어 들어감.

### 6.2 카테고리 변수에 적용 X
`성별`(M/F)을 정규화하면 의미 없음. 수치 컬럼만.

### 6.3 결과 해석 시 원래 단위로
표준화된 가중치는 "표준화된 단위에서"의 영향. 원래 단위 해석은 별도 환산.

### 6.4 0으로 나누기 (편차가 0)
모든 값이 같은 컬럼은 표준편차가 0. NaN 또는 무한대.  
**해결**: 그런 컬럼은 제거.


## 7. 📚 더 알아보기

- `RobustScaler` — 이상치에 강함 (median, IQR 사용)
- `Normalizer` — 행 단위 정규화 (벡터 길이 1)
- `PowerTransformer` — 정규분포에 가깝게 변환
